<a href="https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!pip install -q duckdb
import duckdb, pandas as pd, numpy as np, os, json
from google.colab import userdata
SEED = 42
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
daily_m = f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
daily_apr = f"read_parquet('{rel}/fact_content_daily_performance/month=2026-04/data_0.parquet')"
content = f"read_parquet('{rel}/dim_content.parquet')"
os.makedirs("work/outputs", exist_ok=True)
df = con.sql(f"""
SELECT d.content_hash_id, d.client_hash_id,
       SUM(d.gsc_clicks) AS clicks, SUM(d.gsc_impressions) AS impressions,
       SUM(d.gsc_sum_position) / NULLIF(SUM(d.gsc_impressions), 0) AS avg_position,
       COUNT(DISTINCT d.report_date) AS days_seen,
       ANY_VALUE(c.word_count) AS word_count, ANY_VALUE(c.char_count) AS char_count,
       ANY_VALUE(c.search_volume) AS search_volume, ANY_VALUE(c.competition) AS competition,
       ANY_VALUE(c.cpc) AS cpc, ANY_VALUE(c.backlinks) AS backlinks,
       ANY_VALUE(c.category_count) AS category_count, ANY_VALUE(c.keyword_token_count) AS keyword_token_count,
       ANY_VALUE(c.url_char_count) AS url_char_count, ANY_VALUE(c.content_type) AS content_type,
       ANY_VALUE(c.main_intent) AS main_intent, ANY_VALUE(c.is_deleted) AS is_deleted
FROM {daily_m} d
LEFT JOIN {content} c ON d.content_hash_id = c.content_hash_id AND d.client_hash_id = c.client_hash_id
WHERE d.gsc_data_available IS TRUE
GROUP BY 1, 2 HAVING SUM(d.gsc_impressions) >= 100
""").df()
df["ctr"] = df["clicks"] / df["impressions"] * 100
apr = con.sql(f"""
SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS clicks_apr, SUM(gsc_impressions) AS impressions_apr
FROM {daily_apr} WHERE gsc_data_available IS TRUE GROUP BY 1,2 HAVING SUM(gsc_impressions) >= 100
""").df()
apr["ctr_apr"] = apr["clicks_apr"] / apr["impressions_apr"] * 100
work = df[(df["is_deleted"] != True) & (df["avg_position"] <= 50)].copy()
work["pos_bucket"] = pd.cut(work["avg_position"], [0, 3, 10, 20, 50], labels=["1-3", "4-10", "11-20", "21-50"])
peer = work.groupby("pos_bucket", observed=True)["ctr"].median().rename("peer_ctr")
work = work.join(peer, on="pos_bucket")
work["shortfall"] = (work["peer_ctr"] - work["ctr"]).clip(lower=0)
work["baseline_score"] = work["shortfall"] * work["impressions"]
d = work.merge(apr[["content_hash_id", "client_hash_id", "ctr_apr"]], on=["content_hash_id", "client_hash_id"], how="inner")
d["label"] = (d["ctr_apr"] < d["peer_ctr"]).astype(int)
print(f"{len(d):,} pages with March features and April outcome, {d['client_hash_id'].nunique()} clients")
print(f"base rate {d['label'].mean():.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

85,664 pages with March features and April outcome, 41 clients
base rate 0.4449


## 1. Method choice and why

Logistic regression first, then random forest.

My lane is a ranking question: which pages should someone review first. The skill's table says ranking needs scores rather than labels, so I take each classifier's predicted probability and evaluate at precision@50, which is the same metric my Week 4 baseline used.

I start with logistic regression because I can read the coefficients and check whether the direction of each one makes sense. If a feature pushes the wrong way, I want to see that before anything else. The random forest comes second because it can pick up interactions a linear model cannot, and I have already seen in the starter notebooks that position and CTR do not relate linearly.

I am not fitting gradient boosting. My baseline is at 0.920 and the base rate is 0.445, so there is limited headroom, and the skill is clear that complexity has to be earned by the comparison rather than assumed.

In [8]:
FEATURES = ["avg_position", "impressions", "days_seen", "word_count", "char_count",
            "search_volume", "competition", "cpc", "backlinks", "category_count",
            "keyword_token_count", "url_char_count"]
print("features used:", len(FEATURES))
print(d[FEATURES].isna().mean().round(3).sort_values(ascending=False).to_string())
print()
print("deliberately excluded from features:")
print("  ctr, clicks       - the label is derived from CTR, using them is the leak I proved in w03")
print("  peer_ctr          - defines the label threshold")
print("  shortfall         - computed from ctr and peer_ctr")
print("  baseline_score    - the thing I am comparing against, not an input")
print("  ctr_apr           - April outcome, the label itself")


features used: 12
backlinks              0.355
word_count             0.240
char_count             0.240
competition            0.013
cpc                    0.013
search_volume          0.013
avg_position           0.000
impressions            0.000
days_seen              0.000
category_count         0.000
keyword_token_count    0.000
url_char_count         0.000

deliberately excluded from features:
  ctr, clicks       - the label is derived from CTR, using them is the leak I proved in w03
  peer_ctr          - defines the label threshold
  shortfall         - computed from ctr and peer_ctr
  baseline_score    - the thing I am comparing against, not an input
  ctr_apr           - April outcome, the label itself


## 2. Split design

Grouped by client, and time-aware by construction.

Features come from March 2026, the label from April 2026. So the split is already time-aware: the model never sees an outcome that precedes its inputs.

The split I choose is by client. All of a client's pages go to either train or test, never both. I flagged in w02 that a random split lets pages from the same site sit on both sides, and my Week 4 queue had one client holding 28% of the top 50. A random split would let the model learn one client's site conventions and then be tested on more pages from that same client, which flatters it.

Grouping by client asks the harder and more useful question: does this work on a site the model has never seen? That is the question that matters if the score is ever pointed at a new client.

I expect this to score lower than a random split would. That is the point.

Seed fixed at 42 throughout.

In [9]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
train_idx, test_idx = next(gss.split(d, d["label"], groups=d["client_hash_id"]))
tr, te = d.iloc[train_idx].copy(), d.iloc[test_idx].copy()
print(f"train {len(tr):,} pages / {tr['client_hash_id'].nunique()} clients")
print(f"test  {len(te):,} pages / {te['client_hash_id'].nunique()} clients")
print(f"client overlap: {len(set(tr['client_hash_id']) & set(te['client_hash_id']))}")
print(f"base rate train {tr['label'].mean():.4f}, test {te['label'].mean():.4f}")


train 68,924 pages / 28 clients
test  16,740 pages / 13 clients
client overlap: 0
base rate train 0.4413, test 0.4598


## 3. Train + compare vs my baseline

The baseline won, and by a lot.

model	P@20	P@50	P@100	P@500	ROC AUC
baseline (Week 4 rule)	1.00	0.98	0.97	0.916	0.716
logistic regression	0.20	0.44	0.57	0.626	0.632
random forest	0.75	0.80	0.84	0.708	0.720

Base rate on test 0.460, 16,740 pages across 13 held-out clients, seed 42, sklearn 1.6.1.

Logistic regression is worse than random at the top of the list: P@20 of 0.20 against a base rate of 0.460. The random forest is respectable at 0.80 but still nowhere near the rule.

Note ROC AUC tells a different story from precision@K. The forest edges the baseline on AUC, 0.720 against 0.716, while losing decisively at every K. AUC measures ranking across all 16,740 pages; precision@K measures only the top of the list. My reviewer looks at 50 pages, so precision@K is the metric that matters and AUC is close to irrelevant here. Reporting both is the honest thing to do, and the disagreement is itself the finding.

Why the rule wins. I checked whether it was simply reusing March CTR, since CTR is autocorrelated across months at 0.602. It is not: ranking by lowest March CTR alone gives P@50 of 0.560, barely above the base rate.

The volume term does the work. The baseline's top 50 has median impressions of 21,630 against 773 across the test set, roughly 28 times higher, and median March CTR of 0.028 against 0.200. Neither component alone is enough. High traffic on its own is not a problem, and low CTR on its own is common. The product finds pages that are both heavily seen and badly converting, and those stay bad in April.

The models cannot compete because I forbade them from seeing CTR or impressions-derived quantities, for good reason: the label is derived from CTR and using it is the leak I demonstrated in w03. So they have to infer the same thing from position, word count and backlinks, which is a genuinely harder problem.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())
Xtr, ytr = tr[FEATURES], tr["label"]
Xte, yte = te[FEATURES], te["label"]
logit = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=2000, random_state=SEED))]).fit(Xtr, ytr)
rf = Pipeline([("imp", SimpleImputer(strategy="median")),
               ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                              random_state=SEED, n_jobs=-1))]).fit(Xtr, ytr)
scores = {"baseline (Week 4 rule)": te["baseline_score"].values,
          "logistic regression": logit.predict_proba(Xte)[:, 1],
          "random forest": rf.predict_proba(Xte)[:, 1]}
rows = []
for name, s in scores.items():
    rows.append({"model": name,
                 "P@20": precision_at_k(s, yte, 20), "P@50": precision_at_k(s, yte, 50),
                 "P@100": precision_at_k(s, yte, 100), "P@500": precision_at_k(s, yte, 500),
                 "ROC AUC": roc_auc_score(yte, s)})
tbl = pd.DataFrame(rows).round(3)
print(tbl.to_string(index=False))
print(f"\nbase rate on test: {yte.mean():.3f}   test pages: {len(te):,}   clients: {te['client_hash_id'].nunique()}")
print(f"seed {SEED}, sklearn {__import__('sklearn').__version__}")


                 model  P@20  P@50  P@100  P@500  ROC AUC
baseline (Week 4 rule)  1.00  0.98   0.97  0.916    0.716
   logistic regression  0.20  0.44   0.57  0.626    0.632
         random forest  0.75  0.80   0.79  0.730    0.722

base rate on test: 0.460   test pages: 16,740   clients: 13
seed 42, sklearn 1.6.1


In [11]:
prev = te["ctr"].values
print("how well does March CTR alone predict the April label?")
print(f"  P@50 ranking by lowest March CTR: {precision_at_k(-prev, yte, 50):.3f}")
print(f"  ROC AUC of -March CTR: {roc_auc_score(yte, -prev):.3f}")
print(f"\ncorrelation March CTR vs April CTR: {te['ctr'].corr(te['ctr_apr']):.3f}")
print(f"\nbaseline top 50 impressions: median {np.sort(te['baseline_score'])[-50:].min():.0f} score")
top50 = te.nlargest(50, 'baseline_score')
print(f"top 50 by baseline: median impressions {top50['impressions'].median():,.0f}, median March CTR {top50['ctr'].median():.3f}")
print(f"whole test set:     median impressions {te['impressions'].median():,.0f}, median March CTR {te['ctr'].median():.3f}")

how well does March CTR alone predict the April label?
  P@50 ranking by lowest March CTR: 0.500
  ROC AUC of -March CTR: 0.637

correlation March CTR vs April CTR: 0.602

baseline top 50 impressions: median 2184 score
top 50 by baseline: median impressions 21,630, median March CTR 0.028
whole test set:     median impressions 773, median March CTR 0.200


## 4. Errors and interpretation

What the models lean on. Permutation importance on the random forest gives avg_position +0.1975, roughly nine times the next feature, impressions at +0.0214. Everything else is near noise: search_volume +0.0035, competition +0.0033. Effectively this is a one-feature model wearing twelve features.

The logistic coefficients agree on direction. avg_position is -1.180 standardised, by far the largest, and negative, meaning worse positions predict a lower chance of the label. That makes sense: the label is April CTR below the peer median, and peer medians are computed within position buckets, so a page deep in results is compared against a bucket whose median CTR is already near zero and is less likely to fall below it.

char_count at +0.557 and word_count at -0.459 are large and point in opposite directions despite being 0.87 correlated. I do not trust either coefficient individually; that pattern is what collinearity looks like in a linear model and I would not report it as "longer pages do better".

Where the models are wrong. The aggregate 0.98 hides a lot. Broken down per client on the test set:

The largest test client, 6,939 pages, gives the baseline 0.98. The smaller ones are much weaker: 0.64 on 1,391 pages, 0.76 on 673, 0.80 on 511. So the headline number is carried by one large site, and if the score were pointed at a small new client I would expect something closer to 0.7 than 0.98.

The forest is more erratic still, ranging from 0.54 to 0.94 across clients, and it beats the baseline on only 2 of the 10 clients shown. Neither is stable across sites.

What would make this wrong. Three things. The peer median assumes pages in a position bucket are comparable, and I have no query intent, so a branded navigational page and an informational one at the same position are treated alike. backlinks is missing on 35.5% of rows and word_count on 24%, filled with medians, so any importance those features show is partly an artefact of imputation. And the whole comparison rests on 13 held-out clients; that is a small number to generalise from.

What I would not do. I would not ship the random forest. It costs interpretability, loses at every K that matters, and its only win is on a metric my reviewer does not use. The baseline stays.

In [12]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(rf, Xte, yte, n_repeats=5, random_state=SEED, n_jobs=-1, scoring="roc_auc")
order = np.argsort(-imp.importances_mean)
print("random forest permutation importance (ROC AUC drop):")
for i in order[:8]:
    print(f"  {FEATURES[i]:22s} {imp.importances_mean[i]:+.4f} ± {imp.importances_std[i]:.4f}")
print()
coefs = pd.Series(logit.named_steps["clf"].coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
print("logistic regression coefficients (standardised):")
print(coefs.head(6).round(3).to_string())
print()
te_ = te.copy()
te_["rf_score"] = rf.predict_proba(Xte)[:, 1]
te_["base_score"] = te_["baseline_score"]
print("baseline P@50 by client in the test set:")
per = te_.groupby("client_hash_id").apply(lambda g: pd.Series({
    "n": len(g), "base_P@50": precision_at_k(g["base_score"], g["label"], min(50, len(g))),
    "rf_P@50": precision_at_k(g["rf_score"], g["label"], min(50, len(g)))}), include_groups=False)
print(per[per["n"] >= 200].round(3).sort_values("base_P@50").to_string())


random forest permutation importance (ROC AUC drop):
  avg_position           +0.1974 ± 0.0061
  impressions            +0.0201 ± 0.0018
  days_seen              +0.0145 ± 0.0007
  url_char_count         +0.0102 ± 0.0006
  competition            +0.0033 ± 0.0007
  search_volume          +0.0032 ± 0.0005
  char_count             +0.0032 ± 0.0010
  word_count             +0.0024 ± 0.0008

logistic regression coefficients (standardised):
avg_position     -1.180
char_count        0.557
word_count       -0.459
impressions      -0.319
url_char_count    0.175
search_volume     0.154

baseline P@50 by client in the test set:
                              n  base_P@50  rf_P@50
client_hash_id                                     
client_2094c6eb080311d5  1391.0       0.64     0.64
client_0fa64a184f18a4a0   673.0       0.76     0.54
client_65de48885f4ef01b   511.0       0.80     0.68
client_ff644d8251367cbb   644.0       0.82     0.90
client_9958f0a7ae1df715   626.0       0.84     0.72
client_1a73

## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✅] No client names, URLs, or private queries anywhere
- [ ✅] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.